# Framework CNN-BiLSTM con fusión multimodal para Lengua de Señas Peruana

Clasificación de 34 palabras de la categoría 3 del dataset SignoPeruLSP-ISO. El cuaderno entrena y evalúa la
fusión de RGB y landmarks con un protocolo *cross-subject*:

- **Entrenamiento:** 12 sujetos (2813 muestras).
- **Validación:** sujetos 1, 7 y 8 (988 muestras).
- **Prueba:** sujetos 5 y 14 (587 muestras).

## Antes de ejecutar

El código se descarga de GitHub, pero **los datos no son públicos**. Copia `datos_lsp_colab.zip` en tu Google Drive,
dentro de la carpeta `LSP_Colab`. Después usa **Entorno de ejecución → Ejecutar todas**.

## 1. Descargar el código

In [ ]:
import os
REPO = '/content/lsp-cnn-bilstm'
if not os.path.exists(REPO):
    !git clone -q https://github.com/Mauricio12-ai/lsp-cnn-bilstm.git {REPO}
%cd {REPO}
!git log --oneline -n 1

## 2. Conectar Google Drive y preparar los datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/LSP_Colab/datos_lsp_colab.zip'
assert os.path.exists(ZIP), 'No se encontró ' + ZIP + '. Copia ahí el archivo de datos.'
!unzip -q -n "{ZIP}" -d {REPO}

## 3. Comprobar los datos

In [ ]:
import csv, glob, torch
INDICE = 'datos/indices/union_20261012_153325/indice_comun.csv'
LANDMARKS = 'datos/procesados/landmarks/datos_procesados_20261005_160031/landmarks_32f.npz'
FEATURES = 'datos/caracteristicas/resnet18'
with open(INDICE, encoding='utf-8-sig') as f:
    muestras = sum(1 for _ in csv.DictReader(f))
archivos = len(glob.glob(FEATURES + '/[0-9]*.npz'))
print('PyTorch:', torch.__version__)
print('Muestras del índice:', muestras)
print('Características RGB:', archivos)
assert muestras == 4388 and archivos == 4388, 'Los datos están incompletos.'

## 4. Entrenar la fusión

Las características de ResNet18 ya están calculadas, porque esta red está congelada. Se entrenan las dos BiLSTM y el
clasificador. La mejor época se elige con el F1-macro de validación.

En la computadora original, con 4 hilos y semilla 42, se obtuvo 53.32 % de exactitud y F1-macro de 0.492. En Colab el resultado
puede variar ligeramente, porque cambian el procesador y las versiones de las librerías.

In [ ]:
import time
RUN = 'resultados/fusion/colab_' + time.strftime('%Y%m%d_%H%M%S')
!python codigo/modelos/cnn_bilstm/entrenar_fusion.py --indice "{INDICE}" --landmarks "{LANDMARKS}" --features "{FEATURES}" --salida "{RUN}" --modo fusion --epochs 50 --patience 10 --test-subjects 5 14 --val-subjects 1 7 8 --seed 42 --hilos 4

## 5. Evaluar en los sujetos de prueba

In [ ]:
!python codigo/modelos/cnn_bilstm/evaluar_checkpoint.py --experimento "{RUN}" --indice "{INDICE}" --landmarks "{LANDMARKS}" --features "{FEATURES}"

## 6. Ver los resultados

In [ ]:
import json
carpeta = sorted(glob.glob(RUN + '/test_*'))[-1]
m = json.load(open(carpeta + '/metricas_test.json', encoding='utf-8'))
print(f"Mejor época: {m['best_epoch']}")
print(f"Aciertos: {m['correct']} de {m['samples']}")
print(f"Exactitud: {m['accuracy']:.4f}")
print(f"F1-macro: {m['macro_f1_34']:.4f}")
print(f"Precisión macro: {m['macro_precision_34']:.4f}")
print(f"Recall macro: {m['macro_recall_34']:.4f}")

In [ ]:
from IPython.display import HTML
HTML(open(carpeta + '/matriz_test.html', encoding='utf-8').read())

## 7. Opcional: variantes por modalidad

Cambia `MODO` por `'landmarks'` o `'rgb'` y ejecuta la celda para entrenar y evaluar cada rama por separado.

In [ ]:
MODO = 'landmarks'
RUN2 = f'resultados/colab_{MODO}_' + time.strftime('%Y%m%d_%H%M%S')
!python codigo/modelos/cnn_bilstm/entrenar_fusion.py --indice "{INDICE}" --landmarks "{LANDMARKS}" --features "{FEATURES}" --salida "{RUN2}" --modo {MODO} --epochs 50 --patience 10 --test-subjects 5 14 --val-subjects 1 7 8 --seed 42 --hilos 4
!python codigo/modelos/cnn_bilstm/evaluar_checkpoint.py --experimento "{RUN2}" --indice "{INDICE}" --landmarks "{LANDMARKS}" --features "{FEATURES}"